In [3]:
import cv2
from ultralytics import YOLO
import time

In [4]:
yolo_model = 'yolo26n.pt'
model = YOLO(yolo_model)


In [5]:

import os, random, cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches

ROOT = "dataset"

def load_sample(split="train"):
    split_path = os.path.join(ROOT, split)
    sequences = [s for s in os.listdir(split_path)
                 if os.path.isdir(os.path.join(split_path, s))]

    for _ in range(50):
        seq = random.choice(sequences)
        vis_dir = os.path.join(split_path, seq, "visible")
        if not os.path.exists(vis_dir):
            continue
        jpgs = sorted([f for f in os.listdir(vis_dir) if f.endswith(".jpg")])
        if not jpgs:
            continue
        
        img_name = random.choice(jpgs)
        txt_name = img_name.replace(".jpg", ".txt")
        img_path = os.path.join(vis_dir, img_name)
        txt_path = os.path.join(vis_dir, txt_name)

        img = cv2.imread(img_path)
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        boxes = []
        if os.path.exists(txt_path):
            with open(txt_path) as f:
                for line in f.read().strip().splitlines():
                    parts = line.strip().split()
                    if len(parts) == 5:          
                        cx, cy, bw, bh = map(float, parts[1:])
                        x1 = (cx - bw/2) * w
                        y1 = (cy - bh/2) * h
                        boxes.append((x1, y1, bw*w, bh*h))

        return img, boxes, seq, img_name
    
    return None, [], "?", "?"

fig, axes = plt.subplots(2, 4, figsize=(20, 9))

for ax in axes.flatten():
    img, boxes, seq, fname = load_sample("train")
    if img is None:
        ax.axis("off")
        continue
    
    ax.imshow(img)
    for (x1, y1, bw, bh) in boxes:
        rect = patches.Rectangle(
            (x1, y1), bw, bh,
            linewidth=2, edgecolor='#00FF41', facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(x1+2, y1-6, "UAV", color='#00FF41', fontsize=8,
                fontweight='bold',
                bbox=dict(facecolor='black', alpha=0.6, pad=1, edgecolor='none'))
    
    status = f"✓ {len(boxes)} bbox" if boxes else "✗ no drone"
    color  = '#00FF41' if boxes else '#FF4444'
    ax.set_title(f"{seq[-20:]}\n{fname}  [{status}]",
                 fontsize=7, color=color)
    ax.axis("off")

plt.suptitle("Anti-UAV Dataset — Random Visible Frames", fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig("sample_preview.png", dpi=150, bbox_inches='tight')
plt.show()


<Figure size 2000x900 with 8 Axes>

In [10]:
from ultralytics import YOLO
from pathlib import Path

ROOT = Path(r"C:\Users\ahmed\Desktop\me\me\Coding\HARDCORE\Galatic_Defender\Anti-UAV.yolo26")
modelpath =Path(r"C:\Users\ahmed\Desktop\me\me\Coding\HARDCORE\Galatic_Defender\Anti-UAV.yolo26\runs\anti_uav_yolo26n_localv2\weights\bestcopy.pt")
model = YOLO(modelpath) 

model.train(
    data          = str(ROOT / "data.yaml"),
    epochs        = 50,
    imgsz         = 640,
    copy_paste = 1,   
    batch         = 16,
    device        = 0,
    workers       = 0,
    amp           = True,
    optimizer     = "MuSGD",
    cos_lr=True,       
    lr0 = 0.001,  
    lrf = 0.01,        
    momentum=0.947,  
    flipud       = 0.3,    
    perspective  = 0.001,
    translate= 0.2, 
    shear        = 2.0,    
    erasing      = 0.4,   
    crop_fraction= 1.0,
    hsv_v        = 0.5,  
    hsv_s        = 0.7,    
    warmup_epochs = 3,
    mosaic        = 1.0,
    close_mosaic  = 15,
    scale         = 0.5,
    cutmix = 0.4,
    auto_augment= 'randaugment',
    fliplr        = 0.5,
    degrees       = 10.0,
    patience      = 20,
    save          = True,
    save_period   = 10,
    project       = str(ROOT / "runs"),
    name          = "anti_uav_yolo26n_localv3",
    plots         = True,
    verbose=True
)


WARNING 'crop_fraction' is deprecated and will be removed in the future.
Ultralytics 8.4.41  Python-3.11.0rc2 torch-2.11.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=1, copy_paste_mode=flip, cos_lr=True, cutmix=0.4, data=C:\Users\ahmed\Desktop\me\me\Coding\HARDCORE\Galatic_Defender\Anti-UAV.yolo26\data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.5, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:\Users\ah

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000024CA482FE10>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480